# Phase 00: Revised Project Setup and Data Inventory

This notebook performs only Phase 00 work: path checks, source folder checks,
recursive inventory, revised phase-structure recording, and setup reporting.
It does not perform feature extraction, labeling, ML training, or HDC training.

## 1. Project path check

Confirm the current working directory and prepare output locations under `experiments`.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
EXPECTED_ROOT_NAME = "hdc-vr-pilot"
DATASET_ROOT = PROJECT_ROOT / "vrdataset"
DATA_PACKAGE = DATASET_ROOT / "dataPackage"
STARTER_CODE = DATASET_ROOT / "starterCode"
REFERENCE_DOCS = DATASET_ROOT / "referenceDocuments"

PHASE00 = PROJECT_ROOT / "experiments" / "phase_00_project_setup"
RESULTS_DIR = PHASE00 / "results"
TABLES_DIR = PHASE00 / "tables"
LOGS_DIR = PHASE00 / "logs"
for directory in [RESULTS_DIR, TABLES_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

executed_at_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")
print(f"Current working directory: {PROJECT_ROOT}")
print(f"Expected root name: {EXPECTED_ROOT_NAME}")
print(f"Project root name check: {PROJECT_ROOT.name == EXPECTED_ROOT_NAME}")
print(f"Notebook executed at UTC: {executed_at_utc}")

Current working directory: E:\hdc-vr-pilot
Expected root name: hdc-vr-pilot
Project root name check: True
Notebook executed at UTC: 2026-07-01T17:23:08+00:00


## 2. Required source folder checks

Verify that `vrdataset`, `dataPackage`, `starterCode`, and `referenceDocuments` exist.
The source dataset is read-only for this project.

In [2]:
folder_checks = pd.DataFrame(
    [
        {"folder": "vrdataset", "path": DATASET_ROOT.relative_to(PROJECT_ROOT).as_posix(), "exists": DATASET_ROOT.is_dir()},
        {"folder": "dataPackage", "path": DATA_PACKAGE.relative_to(PROJECT_ROOT).as_posix(), "exists": DATA_PACKAGE.is_dir()},
        {"folder": "starterCode", "path": STARTER_CODE.relative_to(PROJECT_ROOT).as_posix(), "exists": STARTER_CODE.is_dir()},
        {"folder": "referenceDocuments", "path": REFERENCE_DOCS.relative_to(PROJECT_ROOT).as_posix(), "exists": REFERENCE_DOCS.is_dir()},
    ]
)
display(folder_checks)
print(f"Whether vrdataset exists: {DATASET_ROOT.is_dir()}")
print(f"Whether dataPackage exists: {DATA_PACKAGE.is_dir()}")
print(f"Whether starterCode exists: {STARTER_CODE.is_dir()}")
print(f"Whether referenceDocuments exists: {REFERENCE_DOCS.is_dir()}")

if not folder_checks["exists"].all():
    missing = folder_checks.loc[~folder_checks["exists"], "path"].tolist()
    raise FileNotFoundError(f"Missing required source folders: {missing}")
print("Source folder checks passed. No files under vrdataset are modified by this notebook.")

,folder,path,exists
0,vrdataset,vrdataset,True
1,dataPackage,vrdataset/dataPackage,True
2,starterCode,vrdataset/starterCode,True
3,referenceDocuments,vrdataset/referenceDocuments,True


Whether vrdataset exists: True
Whether dataPackage exists: True
Whether starterCode exists: True
Whether referenceDocuments exists: True
Source folder checks passed. No files under vrdataset are modified by this notebook.


## 3. Source role clarification

`starterCode` is mainly an eye-tracking and oculomotor pilot reference. It can help
interpret example processing choices, but it is not the final full multimodal dataset.

`dataPackage` is the main source for the full multimodal thesis experiment. Later phases
should audit and extract features from `vrdataset/dataPackage`, after Phase 01 documents
available modalities, runs, subjects, levels, and quality constraints.

In [3]:
source_roles = pd.DataFrame(
    [
        {
            "source": "vrdataset/starterCode",
            "role": "Eye-tracking and oculomotor pilot reference",
            "use_in_final_experiment": "Reference only; not the final full multimodal dataset",
        },
        {
            "source": "vrdataset/dataPackage",
            "role": "Main source for full multimodal experiment",
            "use_in_final_experiment": "Primary source for Phase 01 audit and Phase 02 feature extraction",
        },
        {
            "source": "vrdataset/referenceDocuments",
            "role": "Documentation and data dictionary source",
            "use_in_final_experiment": "Reference for schema, quality, and data interpretation",
        },
    ]
)
display(source_roles)
print("starterCode is treated as an eye-tracking and oculomotor pilot reference only.")
print("dataPackage is treated as the main source for the full multimodal experiment.")

,source,role,use_in_final_experiment
0,vrdataset/starterCode,Eye-tracking and oculomotor pilot reference,Reference only; not the final full multimodal ...
1,vrdataset/dataPackage,Main source for full multimodal experiment,Primary source for Phase 01 audit and Phase 02...
2,vrdataset/referenceDocuments,Documentation and data dictionary source,"Reference for schema, quality, and data interp..."


starterCode is treated as an eye-tracking and oculomotor pilot reference only.
dataPackage is treated as the main source for the full multimodal experiment.


## 4. Recursive file inventory for `vrdataset`

Scan all files under the read-only source folder and build a full inventory. The requested
target extensions are: csv, mat, xdf, txt, npy, npz, pkl, parquet, py, m, and ipynb.

In [4]:
TARGET_EXTENSIONS = ["csv", "mat", "xdf", "txt", "npy", "npz", "pkl", "parquet", "py", "m", "ipynb"]
TARGET_EXTENSION_SET = {f".{ext}" for ext in TARGET_EXTENSIONS}

def top_area(path: Path) -> str:
    rel = path.relative_to(DATASET_ROOT)
    parts = rel.parts
    if len(parts) >= 2 and parts[0] == "starterCode" and parts[1] == "data_feats":
        return "starterCode/data_feats"
    return parts[0] if parts else "vrdataset"

def folder_depth(path: Path, depth: int = 4) -> str:
    rel_parent = path.parent.relative_to(DATASET_ROOT)
    parts = rel_parent.parts[:depth]
    return "/".join(parts) if parts else "."

all_files = sorted(path for path in DATASET_ROOT.rglob("*") if path.is_file())
all_dirs = sorted(path for path in DATASET_ROOT.rglob("*") if path.is_dir())
records = []
for path in all_files:
    stat = path.stat()
    suffix = path.suffix.lower()
    extension = suffix.lstrip(".") if suffix else "[no_extension]"
    records.append(
        {
            "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
            "dataset_relative_path": path.relative_to(DATASET_ROOT).as_posix(),
            "file_name": path.name,
            "extension": extension,
            "is_target_extension": suffix in TARGET_EXTENSION_SET,
            "dataset_area": top_area(path),
            "analysis_folder": folder_depth(path),
            "size_bytes": stat.st_size,
            "modified_time_utc": datetime.fromtimestamp(stat.st_mtime, timezone.utc).isoformat(timespec="seconds"),
        }
    )

data_inventory = pd.DataFrame.from_records(records).sort_values(["dataset_area", "extension", "relative_path"]).reset_index(drop=True)
inventory_path = RESULTS_DIR / "data_inventory.csv"
data_inventory.to_csv(inventory_path, index=False)

print(f"Number of directories found under vrdataset: {len(all_dirs):,}")
print(f"Number of files found: {len(data_inventory):,}")
print(f"Inventory saved to: {inventory_path.relative_to(PROJECT_ROOT).as_posix()}")
display(data_inventory.head(10))

Number of directories found under vrdataset: 634
Number of files found: 9,022
Inventory saved to: experiments/phase_00_project_setup/results/data_inventory.csv


,relative_path,dataset_relative_path,file_name,extension,is_target_extension,dataset_area,analysis_folder,size_bytes,modified_time_utc
0,vrdataset/LICENSE.txt,LICENSE.txt,LICENSE.txt,txt,True,LICENSE.txt,.,2348,2022-08-24T09:05:10+00:00
1,vrdataset/SHA256SUMS.txt,SHA256SUMS.txt,SHA256SUMS.txt,txt,True,SHA256SUMS.txt,.,1942965,2022-08-25T03:32:34+00:00
2,vrdataset/dataPackage/task-ils/PerfMetrics.csv,dataPackage/task-ils/PerfMetrics.csv,PerfMetrics.csv,csv,True,dataPackage,dataPackage/task-ils,22689,2022-07-18T00:05:28+00:00
3,vrdataset/dataPackage/task-ils/sub-cp003/ses-2...,dataPackage/task-ils/sub-cp003/ses-20210206/le...,sub-cp003_ses-20210206_task-ils_stream-lslhtcv...,csv,True,dataPackage,dataPackage/task-ils/sub-cp003/ses-20210206,576,2022-07-18T00:07:52+00:00
4,vrdataset/dataPackage/task-ils/sub-cp003/ses-2...,dataPackage/task-ils/sub-cp003/ses-20210206/le...,sub-cp003_ses-20210206_task-ils_stream-lslhtcv...,csv,True,dataPackage,dataPackage/task-ils/sub-cp003/ses-20210206,75,2022-07-18T00:07:52+00:00
5,vrdataset/dataPackage/task-ils/sub-cp003/ses-2...,dataPackage/task-ils/sub-cp003/ses-20210206/le...,sub-cp003_ses-20210206_task-ils_stream-lslshim...,csv,True,dataPackage,dataPackage/task-ils/sub-cp003/ses-20210206,21890285,2022-07-18T00:07:52+00:00
6,vrdataset/dataPackage/task-ils/sub-cp003/ses-2...,dataPackage/task-ils/sub-cp003/ses-20210206/le...,sub-cp003_ses-20210206_task-ils_stream-lslshim...,csv,True,dataPackage,dataPackage/task-ils/sub-cp003/ses-20210206,93,2022-07-18T00:07:52+00:00
7,vrdataset/dataPackage/task-ils/sub-cp003/ses-2...,dataPackage/task-ils/sub-cp003/ses-20210206/le...,sub-cp003_ses-20210206_task-ils_stream-lslshim...,csv,True,dataPackage,dataPackage/task-ils/sub-cp003/ses-20210206,4220974,2022-07-18T00:07:52+00:00
8,vrdataset/dataPackage/task-ils/sub-cp003/ses-2...,dataPackage/task-ils/sub-cp003/ses-20210206/le...,sub-cp003_ses-20210206_task-ils_stream-lslshim...,csv,True,dataPackage,dataPackage/task-ils/sub-cp003/ses-20210206,92,2022-07-18T00:07:52+00:00
9,vrdataset/dataPackage/task-ils/sub-cp003/ses-2...,dataPackage/task-ils/sub-cp003/ses-20210206/le...,sub-cp003_ses-20210206_task-ils_stream-lslshim...,csv,True,dataPackage,dataPackage/task-ils/sub-cp003/ses-20210206,30448966,2022-07-18T00:07:52+00:00


## 5. File type summary and folder summary

Summarize requested file extensions and the required source folders separately.

In [5]:
observed_counts = data_inventory["extension"].value_counts().to_dict()
file_type_summary = pd.DataFrame(
    [
        {
            "extension": ext,
            "file_count": int(observed_counts.get(ext, 0)),
            "total_size_bytes": int(data_inventory.loc[data_inventory["extension"] == ext, "size_bytes"].sum()),
        }
        for ext in TARGET_EXTENSIONS
    ]
)
file_type_summary["total_size_mib"] = (file_type_summary["total_size_bytes"] / (1024 ** 2)).round(3)
file_type_summary.to_csv(TABLES_DIR / "file_type_summary.csv", index=False)

folder_specs = {
    "vrdataset/dataPackage": DATA_PACKAGE,
    "vrdataset/starterCode": STARTER_CODE,
    "vrdataset/starterCode/data_feats": STARTER_CODE / "data_feats",
    "vrdataset/referenceDocuments": REFERENCE_DOCS,
}
folder_rows = []
for label, folder in folder_specs.items():
    files = [path for path in folder.rglob("*") if path.is_file()] if folder.exists() else []
    target_files = [path for path in files if path.suffix.lower() in TARGET_EXTENSION_SET]
    folder_rows.append(
        {
            "folder": label,
            "exists": folder.exists(),
            "all_file_count": len(files),
            "target_extension_file_count": len(target_files),
            "total_size_bytes": sum(path.stat().st_size for path in files),
            "total_size_mib": round(sum(path.stat().st_size for path in files) / (1024 ** 2), 3),
        }
    )
folder_summary = pd.DataFrame(folder_rows)
folder_summary.to_csv(TABLES_DIR / "folder_summary.csv", index=False)

print("File type summary:")
display(file_type_summary)
print("Folder summary:")
display(folder_summary)

File type summary:


,extension,file_count,total_size_bytes,total_size_mib
0,csv,9004,47302737654,45111.406
1,mat,0,0,0.000
2,xdf,0,0,0.000
3,txt,2,1945313,1.855
4,npy,0,0,0.000
5,npz,0,0,0.000
6,pkl,0,0,0.000
7,parquet,0,0,0.000
8,py,4,47131,0.045
9,m,1,7647,0.007


Folder summary:


,folder,exists,all_file_count,target_extension_file_count,total_size_bytes,total_size_mib
0,vrdataset/dataPackage,True,9003,9003,47302398864,45111.083
1,vrdataset/starterCode,True,12,12,2953656,2.817
2,vrdataset/starterCode/data_feats,True,1,1,338790,0.323
3,vrdataset/referenceDocuments,True,5,0,322109168,307.187


## 6. Top candidate folders for Phase 01

Rank source folders by file count and size to guide the raw data modality audit.

In [6]:
candidate_source = data_inventory[data_inventory["relative_path"].str.startswith("vrdataset/dataPackage/")].copy()
top_candidate_folders = (
    candidate_source.groupby("analysis_folder", as_index=False)
    .agg(file_count=("relative_path", "count"), total_size_bytes=("size_bytes", "sum"))
    .sort_values(["file_count", "total_size_bytes"], ascending=False)
    .head(15)
)
top_candidate_folders["total_size_mib"] = (top_candidate_folders["total_size_bytes"] / (1024 ** 2)).round(3)
print("Top candidate folders for Phase 01 raw modality audit:")
display(top_candidate_folders)

Top candidate folders for Phase 01 raw modality audit:


,analysis_folder,file_count,total_size_bytes,total_size_mib
31,dataPackage/task-ils/sub-cp037/ses-20210113,240,1563915124,1491.466
26,dataPackage/task-ils/sub-cp031/ses-20211216,240,1490826883,1421.763
29,dataPackage/task-ils/sub-cp035/ses-20220125,240,1474198908,1405.906
20,dataPackage/task-ils/sub-cp025/ses-20210820,240,1439056874,1372.392
34,dataPackage/task-ils/sub-cp042/ses-20220111,240,1431763840,1365.436
24,dataPackage/task-ils/sub-cp029/ses-20220119,240,1427052465,1360.943
35,dataPackage/task-ils/sub-cp043/ses-20220127,240,1407365930,1342.169
28,dataPackage/task-ils/sub-cp033/ses-20211215,240,1406605055,1341.443
23,dataPackage/task-ils/sub-cp028/ses-20211214,240,1376515439,1312.747
10,dataPackage/task-ils/sub-cp014/ses-20211112,240,1374200684,1310.540


## 7. Revised phase structure

Save and display the revised 11-phase thesis workflow.

In [7]:
phase_rows = [
    {"phase_index": 0, "phase_id": "phase_00_project_setup", "phase_title": "Project setup and data inventory", "status": "complete"},
    {"phase_index": 1, "phase_id": "phase_01_raw_data_modality_audit", "phase_title": "Raw data audit and modality inventory", "status": "not_started"},
    {"phase_index": 2, "phase_id": "phase_02_full_multimodal_feature_extraction", "phase_title": "Full multimodal feature extraction", "status": "not_started"},
    {"phase_index": 3, "phase_id": "phase_03_multimodal_dataset_labeling", "phase_title": "Multimodal dataset construction and four-class labeling", "status": "not_started"},
    {"phase_index": 4, "phase_id": "phase_04_ml_baseline_four_class", "phase_title": "Traditional ML four-class baseline", "status": "not_started"},
    {"phase_index": 5, "phase_id": "phase_05_basic_hdc_four_class", "phase_title": "Basic HDC four-class classification", "status": "not_started"},
    {"phase_index": 6, "phase_id": "phase_06_hdc_classifier_screening", "phase_title": "HDC classifier screening", "status": "not_started"},
    {"phase_index": 7, "phase_id": "phase_07_single_modality_contribution", "phase_title": "Single-modality contribution analysis", "status": "not_started"},
    {"phase_index": 8, "phase_id": "phase_08_multimodal_fusion", "phase_title": "Multimodal fusion strategy comparison", "status": "not_started"},
    {"phase_index": 9, "phase_id": "phase_09_robustness_missing_modality", "phase_title": "Robustness and missing-modality analysis", "status": "not_started"},
    {"phase_index": 10, "phase_id": "phase_10_onlinehd_lsl_simulation", "phase_title": "OnlineHD and LSL simulation", "status": "not_started"},
]
revised_phase_structure = pd.DataFrame(phase_rows)
phase_structure_path = RESULTS_DIR / "revised_phase_structure.csv"
revised_phase_structure.to_csv(phase_structure_path, index=False)
print(f"Revised phase structure saved to: {phase_structure_path.relative_to(PROJECT_ROOT).as_posix()}")
display(revised_phase_structure)

Revised phase structure saved to: experiments/phase_00_project_setup/results/revised_phase_structure.csv


,phase_index,phase_id,phase_title,status
0,0,phase_00_project_setup,Project setup and data inventory,complete
1,1,phase_01_raw_data_modality_audit,Raw data audit and modality inventory,not_started
2,2,phase_02_full_multimodal_feature_extraction,Full multimodal feature extraction,not_started
3,3,phase_03_multimodal_dataset_labeling,Multimodal dataset construction and four-class...,not_started
4,4,phase_04_ml_baseline_four_class,Traditional ML four-class baseline,not_started
5,5,phase_05_basic_hdc_four_class,Basic HDC four-class classification,not_started
6,6,phase_06_hdc_classifier_screening,HDC classifier screening,not_started
7,7,phase_07_single_modality_contribution,Single-modality contribution analysis,not_started
8,8,phase_08_multimodal_fusion,Multimodal fusion strategy comparison,not_started
9,9,phase_09_robustness_missing_modality,Robustness and missing-modality analysis,not_started


## 8. Save setup report and log

Save a JSON setup report and a plain-text Phase 00 log. These files record what was
checked and where outputs were written.

In [8]:
report = {
    "executed_at_utc": executed_at_utc,
    "project_root": str(PROJECT_ROOT),
    "data_safety": "vrdataset treated as read-only; generated outputs saved under experiments",
    "source_roles": {
        "starterCode": "eye-tracking and oculomotor pilot reference only",
        "dataPackage": "main source for full multimodal experiment",
        "referenceDocuments": "documentation and data dictionary source",
    },
    "folder_checks": folder_checks.to_dict(orient="records"),
    "total_directories_under_vrdataset": len(all_dirs),
    "total_files_under_vrdataset": len(data_inventory),
    "target_extensions": TARGET_EXTENSIONS,
    "file_type_summary": file_type_summary.to_dict(orient="records"),
    "folder_summary": folder_summary.to_dict(orient="records"),
    "top_candidate_folders_for_phase_01": top_candidate_folders.to_dict(orient="records"),
    "outputs": {
        "data_inventory": "experiments/phase_00_project_setup/results/data_inventory.csv",
        "revised_phase_structure": "experiments/phase_00_project_setup/results/revised_phase_structure.csv",
        "project_setup_report": "experiments/phase_00_project_setup/results/project_setup_report.json",
        "file_type_summary": "experiments/phase_00_project_setup/tables/file_type_summary.csv",
        "folder_summary": "experiments/phase_00_project_setup/tables/folder_summary.csv",
        "phase_00_log": "experiments/phase_00_project_setup/logs/phase_00_log.txt",
    },
}
report_path = RESULTS_DIR / "project_setup_report.json"
with report_path.open("w", encoding="utf-8") as handle:
    json.dump(report, handle, indent=2)

log_lines = [
    "Phase 00 revised project setup and inventory log",
    f"Executed at UTC: {executed_at_utc}",
    f"Project root: {PROJECT_ROOT}",
    "Data safety: vrdataset treated as read-only",
    "starterCode role: eye-tracking and oculomotor pilot reference only",
    "dataPackage role: main source for full multimodal experiment",
    f"Total files found under vrdataset: {len(data_inventory):,}",
    f"Data inventory: {inventory_path.relative_to(PROJECT_ROOT).as_posix()}",
    f"Revised phase structure: {phase_structure_path.relative_to(PROJECT_ROOT).as_posix()}",
    f"Project setup report: {report_path.relative_to(PROJECT_ROOT).as_posix()}",
    "No feature extraction, labeling, ML training, or HDC training was performed.",
]
log_path = LOGS_DIR / "phase_00_log.txt"
log_path.write_text(chr(10).join(log_lines) + chr(10), encoding="utf-8")

print(f"Project setup report saved to: {report_path.relative_to(PROJECT_ROOT).as_posix()}")
print(f"Phase 00 log saved to: {log_path.relative_to(PROJECT_ROOT).as_posix()}")
print("Phase 00 complete. No feature extraction, labeling, ML training, or HDC training was performed.")

Project setup report saved to: experiments/phase_00_project_setup/results/project_setup_report.json
Phase 00 log saved to: experiments/phase_00_project_setup/logs/phase_00_log.txt
Phase 00 complete. No feature extraction, labeling, ML training, or HDC training was performed.
